# Mehrotra, Levenstein et al. (2024): the PoSub LFP claims, on dandiset 000939

Paper: Mehrotra, Levenstein et al. (2024), Current Biology 34, 3043-3054,
[doi:10.1016/j.cub.2024.05.048](https://doi.org/10.1016/j.cub.2024.05.048). The registry's target is
dandiset 000987 (`reproduce_mehrotra_2024.ipynb`); these claims use the paper's PoSub data, deposited
as DANDI:000939, and are counted apart (`data: elsewhere`).

Dataset: [DANDI:000939](https://dandiarchive.org/dandiset/000939/0.261016.2310), version
`0.261016.2310` (the most recent published version; the paper cites none; see `posub_000939.ipynb`
for the choice and for the 16 sessions, taken from the event files of the paper's code at commit
4ee34f3). The LFP (1,250 Hz, 64 channels) is streamed for the NREM epochs within the home cage only.

Claims file: `claims/000987/10.1016-j.cub.2024.05.048.yaml`. The paper has no code for Figure 2;
every method detail below is from the STAR Methods, cited in the docstring of
`mehrotra.lfp_event_averages`, or one of the readings the claims file names.

| Claim | Here |
|---|---|
| fig2f-gamma-travels | tested |
| figs2d-lfp-spike-agree | tested |
| figs2e-speed-similar | tested |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, pearsonr, wilcoxon, mannwhitneyu, linregress
import pynapple as nap
import mehrotra as m

print("pynapple", nap.__version__, "| dandiset", m.POSUB_DANDISET, m.POSUB_VERSION)
S = m.load_all_posub()
assets = {a["path"].split("/")[0].replace("sub-", ""): a for a in m.list_assets(m.POSUB_DANDISET, m.POSUB_VERSION)}
# DOWN and UP states: the paper's event files, except A3708-200317b, whose event file does not share
# the clock of the spikes (posub_000939.ipynb); its states are re-detected from the spikes.
EV = {}
for s in S:
    if s["session"] == "A3708-200317b":
        EV[s["session"]] = m.detect_down_states(s["spikes"], s["nrem_hc"], restrict_up=True)
    else:
        EV[s["session"]] = (s["down"], s["up"])
print(len(S), "sessions")

## Data check and event-averaged high gamma power

For each session, high gamma power is averaged per channel around the DU transitions (UP starts),
the UD transitions (DOWN starts) and the delta peaks, under both power readings. Faulty channels
(`is_faulty`) are left out. As a check of the clock shared by the LFP and the spike-derived events,
the rise of power after DU (mean over 0 to 100 ms minus -100 to 0 ms, all channels) is compared with
the same contrast around the DU times shifted by 0.5 s.

In [ ]:
import time
LF = {}
rows = []
for s in S:
    t0 = time.time()
    down, up = EV[s["session"]]
    ev = {"DU": up["start"], "UD": down["start"], "DU_shift_0p5s": up["start"] + 0.5}
    a_ = assets[s["session"].split("-")[0]]
    r = m.lfp_event_averages(a_, s["nrem_hc"], ev)
    lfp_s = m.open_asset(a_)["processing/ecephys/LFP/LFP/data"].shape[0] / m.LFP_FS
    last_spike = max(s["spikes"][k].t[-1] for k in s["spikes"].keys() if len(s["spikes"][k]))
    LF[s["session"]] = r
    lags = r["lags"]
    def contrast(key):
        a = r["avg"][key]
        return float(a[(lags >= 0) & (lags < 0.1)].mean() / a[(lags >= -0.1) & (lags < 0)].mean())
    rows.append(dict(session=s["session"], lfp_duration_s=round(lfp_s), last_spike_s=round(last_spike),
                     channels_kept=len(r["channels"]), delta_channel=r["delta_channel"],
                     n_DU=r["n"][("abs_hilbert", "DU")], n_UD=r["n"][("abs_hilbert", "UD")],
                     n_delta_peaks=r["n"][("abs_hilbert", "delta_peak")],
                     delta_peaks_per_min=round(60 * len(r["delta_peaks"]) / s["nrem_hc"].tot_length(), 2),
                     power_ratio_after_DU=round(contrast(("abs_hilbert", "DU")), 3),
                     power_ratio_after_DU_shifted=round(contrast(("abs_hilbert", "DU_shift_0p5s")), 3),
                     seconds=round(time.time() - t0)))
LC = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(LC.to_string())

## Unit delays (as fig1g-dv-sequence-du)

The per-session Kendall tau between the UP onset delay of EX units and position, as in
`posub_000939.ipynb`, recomputed here for figs2d-lfp-spike-agree and figs2e-speed-similar.
Position is the probe `rel_y` (um, larger is dorsal), so dorsal-first gives a negative tau.

In [ ]:
unit_rows = []
for s in S:
    down, up = EV[s["session"]]
    r = m.pmr_onset(m.du_peth(s["spikes"], up))
    u = s["units"].copy(); u["onset"] = r.onset.values; u["entered"] = r.entered.values
    e = u[u.ex & u.entered]
    t, p = kendalltau(e.onset, e.position)
    lr = linregress(e.position / 1000.0, e.onset)          # delay (s) on depth (mm)
    lr2 = linregress(e.onset, e.position / 1000.0)         # depth (mm) on delay (s)
    unit_rows.append(dict(session=s["session"], n=len(e), tau=t, p=p,
                          speed_inverse_slope=abs(1 / lr.slope), speed_slope_depth_on_delay=abs(lr2.slope)))
UT = pd.DataFrame(unit_rows).set_index("session")
print(UT.round(4).to_string())

## Claim fig2f-gamma-travels

> The rise in high gamma (70 to 150 Hz) power at UP onset travels along the DV axis, while its drop
> at the UD transition is synchronous.

Criterion: per channel, the event-averaged high gamma power; UP onset delay is the time of maximal
derivative; per session, Kendall tau of delay against channel depth; DU coefficients must differ
from 0 (two-sided one-sample Wilcoxon, p < 0.05) and UD must not, under all readings: (1) power as
the absolute Hilbert transform or the square; (2) aligned to DU or to the delta peak; (3) the UD
measure as the time of maximal negative derivative around the DOWN start or the delta peak; (4) the
slope searched 0 to 150 ms after the event or -250 to 250 ms. Blocked as methods_underspecified when
the readings disagree.

For the UD measure, the window "0 to 150 ms after the event" puts the search after the drop, which
precedes the DOWN start; it is run as written, and the mirror window, -150 to 0 ms, is run too.

In [ ]:
def lfp_taus(power, event, window, sign):
    out = []
    for s in S:
        r = LF[s["session"]]
        d = m.max_slope_time(r["avg"][(power, event)], r["lags"], window, sign)
        if np.unique(d).size > 1:
            t, p = kendalltau(d, r["rel_y"])
        else:
            t, p = np.nan, np.nan
        out.append(dict(session=s["session"], tau=t, p=p, delay_min=d.min(), delay_max=d.max(), delays=d))
    return pd.DataFrame(out).set_index("session")

READ = []
TAUS = {}
for power in ("abs_hilbert", "square"):
    for event in ("DU", "delta_peak"):
        for wname, w in (("0 to 150 ms", (0.0, 0.15)), ("-250 to 250 ms", (-0.25, 0.25))):
            du = lfp_taus(power, event, w, +1)
            TAUS[(power, event, wname)] = du
            ud_event = "UD" if event == "DU" else "delta_peak"
            for uw_name, uw in ((wname, w), ("-150 to 0 ms (mirror)", (-0.15, 0.0))) if wname == "0 to 150 ms" else ((wname, w),):
                ud = lfp_taus(power, ud_event, uw, -1)
                dut, udt = du.tau.dropna(), ud.tau.dropna()
                w_du = wilcoxon(dut) if len(dut) > 5 else None
                w_ud = wilcoxon(udt) if len(udt) > 5 else None
                mw = mannwhitneyu(dut, udt) if len(udt) > 5 else None
                READ.append(dict(power=power, event=event, window=wname, ud_window=uw_name,
                                 du_sessions=len(dut), du_median=round(dut.median(), 3), du_negative=int((dut < 0).sum()),
                                 W_du=w_du.statistic if w_du else np.nan, p_du=w_du.pvalue if w_du else np.nan,
                                 ud_sessions=len(udt), ud_median=round(udt.median(), 3) if len(udt) else np.nan,
                                 W_ud=w_ud.statistic if w_ud else np.nan, p_ud=w_ud.pvalue if w_ud else np.nan,
                                 U=mw.statistic if mw else np.nan, p_U=mw.pvalue if mw else np.nan,
                                 du_differs=bool(w_du and w_du.pvalue < 0.05),
                                 ud_not=bool(w_ud and w_ud.pvalue >= 0.05)))
R2 = pd.DataFrame(READ)
print(R2.to_string())
print("\npaper: DU W = 21, p = 0.013; UD W = 68, p = 1; DU against UD U = 73, p = 0.04")
print("readings with DU differing from 0:", int(R2.du_differs.sum()), "of", len(R2),
      "| with UD not differing (where testable):", int(R2.ud_not.sum()), "of", int(R2.ud_sessions.gt(5).sum()))

In [ ]:
main = TAUS[("abs_hilbert", "DU", "0 to 150 ms")]
print(main[["tau", "p", "delay_min", "delay_max"]].round(4).to_string())
s0 = S[1]["session"]; r = LF[s0]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
a = r["avg"][("abs_hilbert", "DU")]
order = np.argsort(-r["rel_y"])
ax[0].imshow((a / a.mean(0)).T[order], aspect="auto", extent=[r["lags"][0], r["lags"][-1], len(order), 0], vmin=0.5, vmax=1.5)
ax[0].set_xlim(-0.25, 0.25); ax[0].set_xlabel("time from DU (s)"); ax[0].set_ylabel("channel, dorsal at top")
ax[0].set_title(f"{s0}: high gamma (|Hilbert|) / channel mean, colour 0.5 to 1.5")
d = main.loc[s0, "delays"]
ax[1].scatter(1000 * d, r["rel_y"], s=8); ax[1].set_xlabel("time of maximal slope (ms)"); ax[1].set_ylabel("rel_y (um)")
plt.tight_layout(); plt.show()
print("normalized power range in the panel:", round(float((a / a.mean(0)).min()), 3), "to", round(float((a / a.mean(0)).max()), 3))

**Result: not reproduced.** No reading gives both parts of the claim. Aligned to DU, as the Methods
align it, the time of maximal slope of high gamma power depends on channel position (median tau
-0.659 to -0.616, negative in 12 or 13 sessions; one-sample Wilcoxon p = 0.0017 to 0.0034), but so
does the time of maximal negative slope before the DOWN start (median tau -0.799 to -0.744, p = 3.1e-5
to 1.2e-4), so the drop is not synchronous by this measure. Aligned to the delta peak, the rise does
not depend on position (p = 0.39 to 0.60). With the UD window written in the criterion, 0 to 150 ms
after the DOWN start, the delays are constant in 12 of 16 sessions and the UD test cannot be run. In
A3705-200306, A3708-200317b and A3711-200810b high gamma power does not rise after DU (ratio 1.000 to
1.045, against 1.003 to 1.048 for times shifted by 0.5 s), so in these sessions the LFP may not share
the clock of the spikes; they are kept, as the paper's procedure would keep them.

## Claim figs2d-lfp-spike-agree

> Per session, the delay-depth correlation computed from units agrees with that computed from high
> gamma LFP power (Pearson R = 0.62, n = 16, p = 0.01).

Criterion: the per-session DU Kendall coefficients of the units against those of the LFP, under
each reading of fig2f; two-sided Pearson test. Reproduced when R is within 0.062 of 0.62 and
p < 0.05 under at least one reading and positive with p < 0.05 under all; blocked as
methods_underspecified when the readings disagree on sign or significance.

In [ ]:
rows = []
for key, du in TAUS.items():
    j = du[["tau"]].merge(UT[["tau"]].rename(columns={"tau": "tau_units"}), left_index=True, right_index=True)
    j = j.dropna(subset=["tau", "tau_units"])
    r, p = pearsonr(j.tau_units, j.tau)
    rows.append(dict(power=key[0], event=key[1], window=key[2], sessions=len(j), R=round(r, 4), p=p,
                     within_tol=abs(r - 0.62) <= 0.062, positive_p05=bool(r > 0 and p < 0.05)))
R3 = pd.DataFrame(rows)
print(R3.to_string())
print("readings positive with p < 0.05:", int(R3.positive_p05.sum()), "of", len(R3),
      "| within tolerance and p < 0.05:", int((R3.within_tol & (R3.p < 0.05)).sum()))

**Result: blocked, methods underspecified.** The per-session unit and LFP coefficients correlate
positively with p < 0.05 under 6 of 8 readings (R = 0.506 to 0.761), and within 0.062 of 0.62 under one
(absolute Hilbert transform, DU, -250 to 250 ms: R = 0.590, p = 0.016). Under the two delta-peak
readings with the absolute Hilbert transform they do not reach p < 0.05 (R = 0.412, p = 0.13; R = 0.462,
p = 0.071). The criterion blocks the claim when the readings disagree on significance.

## Claim figs2e-speed-similar

> For sessions with a significant delay-depth correlation, the speed of the traveling UP onset is
> similar from spikes and from LFP (6.72 +/- 4.08 against 9.89 +/- 5.51 mm/s, Mann-Whitney U = 58,
> n = 13 and 15, p = 0.072).

Criterion: a claim of absence, judged by the two-sided Mann-Whitney U test at p < 0.05; speed from
the slope of the delay-depth relation in sessions whose Kendall correlation is significant; speed
as the inverse of the regression slope of delay on depth, or as the slope of depth on delay; both
are run. Depth in mm (rel_y / 1000), delay in s. Expected values are compared in mm/s, and a match
only after a factor of 1000 is reported as the um/s reading. The LFP side is run under each reading
of fig2f.

In [ ]:
rows = []
us = UT[UT.p < 0.05]
for key, du in TAUS.items():
    sig = du[du.p < 0.05]
    for sp_name in ("speed_inverse_slope", "speed_slope_depth_on_delay"):
        lfp_speed = []
        for sname in sig.index:
            r = LF[sname]
            d = sig.loc[sname, "delays"]
            y = r["rel_y"] / 1000.0
            if sp_name == "speed_inverse_slope":
                lfp_speed.append(abs(1 / linregress(y, d).slope))
            else:
                lfp_speed.append(abs(linregress(d, y).slope))
        lfp_speed = np.array(lfp_speed)
        u_speed = us[sp_name].values
        mw = mannwhitneyu(u_speed, lfp_speed)
        rows.append(dict(power=key[0], event=key[1], window=key[2], speed=sp_name, n_units=len(u_speed), n_lfp=len(lfp_speed),
                         units_mean=round(u_speed.mean(), 2), units_sd=round(u_speed.std(ddof=1), 2),
                         lfp_mean=round(lfp_speed.mean(), 2), lfp_sd=round(lfp_speed.std(ddof=1), 2),
                         U=mw.statistic, p=mw.pvalue, no_difference=mw.pvalue >= 0.05,
                         units_within=abs(u_speed.mean() - 6.72) <= 4.08, lfp_within=abs(lfp_speed.mean() - 9.89) <= 5.51))
R4 = pd.DataFrame(rows)
print(R4.to_string())
print("\nunit sessions with a significant delay-depth correlation:", len(us), "(paper 13)")

**Result: not reproduced, resting on an assumption.** 13 sessions have a significant delay-depth
correlation of units, as in the paper. Speed as the slope of depth on delay gives 6.97 +/- 4.38 mm/s
from units, close to the paper's 6.72 +/- 4.08, and 17.9 to 41.0 mm/s from LFP under the eight readings
(paper 9.89 +/- 5.51), different from the units under every reading (Mann-Whitney p = 1.6e-5 to 0.028).
Speed as the inverse of the slope of delay on depth gives 37.3 +/- 18.0 mm/s from units, far from the
paper's, and no difference under 6 of 8 LFP readings. The outcome rests on the speed definition that
gives the paper's unit speeds.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig2f-gamma-travels | not reproduced |
| figs2d-lfp-spike-agree | blocked, methods underspecified (6 of 8 readings hold) |
| figs2e-speed-similar | not reproduced, resting on an assumption |

How the analysis differs from the paper's. The paper has no code for Figure 2. Power is taken from
the filtered LFP in two ways, events are DU or delta peaks, the UD measure is the time of maximal
negative slope, and the slope is searched in two windows, all as the claims file names them; the
delta channel is the middle channel of the probe, and faulty channels are left out. LFP is read only
for NREM within the home cage epochs, and the DOWN and UP states are those of `posub_000939.ipynb`
(the paper's event files, with A3708-200317b re-detected).

What was found about the dataset (numbers above):

- In A3705-200306 the LFP covers 11,168 s while the spikes run to 16,199 s, and in A3711-200810b
  9,499 s against 12,177 s.
- In three sessions (A3705-200306, A3708-200317b, A3711-200810b) high gamma power does not rise after
  the DU transitions of the spikes, which suggests that their LFP and spikes do not share a clock.